# 🎬 Worker VIDEO — VideoClip Creator

**ANTES DE EJECUTAR (solo la 1ª vez):** menú `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

**Después:** pulsa **`Ctrl+F9`** (o `Entorno de ejecución → Ejecutar todas`) y deja esta pestaña abierta.

Cuando veas `🟢 WORKER ACTIVO` al final, este worker ya se habrá **auto-registrado** en tu orquestador: vuelve a la página Lanzador y verás el ✅.

> ⏱ La primera ejecución tarda 5-10 min (descarga del modelo). Si Colab se desconecta, vuelve a pulsar `Ctrl+F9` (el worker se re-registrará solo con su nueva URL).

In [ ]:
# CELDA 1 · Instalar dependencias (2-3 min)
!pip install -q --upgrade diffusers transformers accelerate sentencepiece \
    imageio imageio-ffmpeg fastapi "uvicorn[standard]" nest-asyncio pillow
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /content/cloudflared && chmod +x /content/cloudflared
print('✅ instalación lista')

In [ ]:
# CELDA 2 · Cargar LTX-Video (modelo open source de Lightricks)
import torch
from diffusers import LTXImageToVideoPipeline

pipe = LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()   # necesario para caber en la T4 (15 GB)
print('✅ modelo cargado')

In [ ]:
# CELDA 3 · Servidor del worker (endpoint /video)
import base64, io, threading, time
import nest_asyncio, uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from PIL import Image
from diffusers.utils import export_to_video

TOKEN = 'b794a2e6f1c3d5b8a0e4f7c2d9b1a3e6f8d0c5b4a2e7f9d1c3b6a8e0f4d7dbb0'   # inyectado por generar_notebooks.py
app = FastAPI()

@app.get('/ping')
def ping():
    return {'ok': True, 'rol': 'video'}

@app.post('/video')
def video(body: dict, x_token: str = Header(default='')):
    if x_token != TOKEN:
        raise HTTPException(401, 'token inválido')
    prompt   = body['prompt'] + ', cinematic, smooth motion, high quality'
    segundos = max(2.0, min(float(body.get('segundos', 5)), 8.0))
    fps      = 24
    frames   = int(segundos * fps) // 8 * 8 + 1   # LTXV exige frames ≡ 1 (mod 8)
    img = Image.open(io.BytesIO(base64.b64decode(body['imagen_b64']))).convert('RGB')
    img.thumbnail((768, 768))
    out = pipe(image=img, prompt=prompt, num_frames=frames, frame_rate=fps,
               num_inference_steps=25, guidance_scale=3.0).frames[0]
    path = f'/content/out_{int(time.time()*1000)}.mp4'
    export_to_video(out, path, fps=fps)
    return Response(content=open(path, 'rb').read(), media_type='video/mp4')

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8080), daemon=True).start()
time.sleep(5)
print('✅ servidor del worker corriendo')

In [ ]:
# CELDA 4 · Túnel público + AUTO-REGISTRO en el orquestador
import subprocess, re, time, requests

proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8080'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(90):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', proc.stdout.readline())
    if m:
        url = m.group(0)
        break
assert url, 'No se pudo crear el túnel: reintenta esta celda'
print('🌐 URL pública del worker:', url)

API = 'https://api.ai-producer-2qd.pages.dev'
for intento in range(30):
    try:
        r = requests.post(API + '/api/workers/registrar',
                          json={'rol': 'video', 'url': url},
                          headers={'X-Token': TOKEN}, timeout=20)
        print('✅ Auto-registrado en el orquestador:', r.json())
        break
    except Exception as e:
        print(f'  reintentando registro ({intento+1}/30)...', e)
        time.sleep(5)
else:
    print('❌ No se pudo registrar: revisa PUBLIC_API_URL y WORKER_TOKEN')

print()
print('🟢 WORKER ACTIVO. Vuelve al Lanzador: verás el check verde ✅')
print('   NO cierres esta pestaña mientras generas videoclips.')
while True:
    time.sleep(60)
